# Neural networks, perceptron, backpropagation, optimization, autoencoders, and VAE

**Rubric weight: 10%.**

Starter notebook. Complete the analysis below.

- Explain the perceptron, backpropagation, and optimization.
- Describe autoencoder latent representations and the probabilistic view of VAEs.
- Connect these concepts to the project and document relevant experiments.


In [ ]:
from pathlib import Path
import sys
import random
import numpy as np
import pandas as pd

# Make the local src package available from the project root or notebooks folder.
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "src" / "__init__.py").is_file():
    raise RuntimeError("Start Jupyter from the project root or notebooks folder.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import SEED, RAW_DATA_PATH, CLEAN_DATA_PATH, RESULTS_DIR
from src import models, evaluation, visualization
from src.preprocessing import cleaning, splitting

random.seed(SEED)
np.random.seed(SEED)
DATA_PATH = CLEAN_DATA_PATH
# Load data when needed, checking encoding and data types.
# Pass random_state=SEED to estimators that support it.
# Add reusable functions in src/ and call them here, e.g. cleaning.your_function(...).


## **1. Question of Interest**

## **2. Database Loading**

## **3. Perceptron**

**1. What is a Perceptron?**

A perceptron is the basic computational unit of an artificial neural network. It receives several input values, multiplies them by weights, adds a bias, and applies an activation function to produce an output.
How does it work?

Mathematically, a neuron computes:

$$
\boxed{\hat y=f\left(\sum_{j=1}^{p}w_jx_j+b\right)}
$$

Where:
- $x_j$: Input feature.
- $w_j$: Weight assigned to each feature.
- $b$: Bias.
- $f$: Activation function.
- $\hat y$: Predicted output.


**2. Numerical example**

Suppose a neuron receives two inputs:
$$
x_1=2,\quad x_2=3
$$

With weights and bias:

$$
w_1=0.5,\quad w_2=-0.2,\quad b=0.1
$$

First, calculate the weighted sum:

$$
z=(0.5)(2)+(-0.2)(3)+0.1=0.5
$$

Then apply an activation function, such as ReLU:

$$
f(z)=\max(0,z)
$$

Therefore:
$$
\boxed{\hat y=0.5}
$$

![Perceptron Architecture](../results/07_autoencoder_vae/figures/perceptron.png)

## **4. Backpropagation**

**Backpropagation** is an algorithm used to calculate how each weight and bias in a neural network contributes to the prediction error.

While the **forward pass** produces a prediction, backpropagation calculates the gradients needed to improve that prediction.

It uses the **chain rule of calculus** to propagate information about the error backward through the network.

**1. How does it work?**

A neural network learns through three main stages:

1. **Forward pass:** Calculate the prediction.
2. **Loss calculation:** Measure the prediction error.
3. **Backward pass:** Calculate the gradients.

**Step 1 — Forward propagation**

The inputs pass through the neurons, producing a prediction:

$$
\hat y=f(wx+b)
$$

**Step 2 — Calculate the loss**

We measure the difference between the prediction and the actual value. For example, using squared error:

$$
L=(y-\hat y)^2
$$

**Step 3 — Backpropagation**

We calculate how the loss changes with respect to each parameter:

$$
\boxed{\frac{\partial L}{\partial w},\quad \frac{\partial L}{\partial b}}
$$

These derivatives are called **gradients**.

A positive gradient indicates that increasing the parameter would locally increase the loss, while a negative gradient indicates the opposite.

**2. Why is the chain rule important?**

Consider a single neuron:

$$
z=wx+b,\qquad \hat y=f(z)
$$

The loss depends on the weight indirectly:

$$
w\rightarrow z\rightarrow \hat y\rightarrow L
$$

Therefore, we use the chain rule:

$$
\boxed{
\frac{\partial L}{\partial w}
=
\frac{\partial L}{\partial \hat y}
\cdot
\frac{\partial \hat y}{\partial z}
\cdot
\frac{\partial z}{\partial w}
}
$$

This allows us to calculate how changing a weight affects the final error, even when the network contains multiple layers.

**3. Numerical example**

Suppose we have a neuron with a linear activation function:

$$
x=2,\quad w=0.5,\quad b=0,\quad y=3
$$

The prediction is:

$$
\hat y=wx+b=(0.5)(2)=1
$$

The squared error is:

$$
L=(3-1)^2=4
$$

Now calculate the gradient:

$$
\frac{\partial L}{\partial w}
=2(\hat y-y)x
$$

$$
=2(1-3)(2)=\boxed{-8}
$$

The negative gradient tells us that increasing the weight will reduce the loss locally.

**4. How are the weights updated?**

Backpropagation only **calculates the gradients**. An optimization algorithm, such as gradient descent or Adam, uses those gradients to update the parameters.

For gradient descent:

$$
\boxed{w_{\text{new}}=w_{\text{old}}-\eta\frac{\partial L}{\partial w}}
$$

Where $\eta$ is the learning rate.

For example, with $\eta=0.1$:

$$
w_{\text{new}}=0.5-0.1(-8)=1.3
$$

The new prediction becomes:

$$
\hat y=(1.3)(2)=2.6
$$

And the new loss is:

$$
L=(3-2.6)^2=\boxed{0.16}
$$

| Metric | Before update | After update |
|---|---|---|
| Weight | 0.5 | 1.3 |
| Prediction | 1.0 | 2.6 |
| Squared error | 4.00 | 0.16 |

The error decreased after one weight update.

**The key distinction:** Backpropagation computes the gradients; optimization uses those gradients to change the weights and minimize the loss.

![Backpropagation Pipeline](../results/07_autoencoder_vae/figures/backpropagation.png)

## **5. Optimization**

**Optimization** is the process of adjusting the weights and biases of a neural network to minimize a loss function.

While **backpropagation calculates the gradients**, optimization uses those gradients to update the model parameters and reduce prediction errors.

The objective is to find the parameters that minimize the loss:

$$
\boxed{\theta^*=\arg\min_\theta L(\theta)}
$$

Where:

- $\theta$: Model parameters (weights and biases).
- $L(\theta)$: Loss function.
- $\theta^*$: Parameters that minimize the loss.

**1. Gradient Descent**

**Gradient Descent** is a fundamental optimization algorithm that updates parameters in the opposite direction of the gradient, since the gradient indicates the direction of greatest local increase in the loss.

The update rule is:

$$
\boxed{\theta_{t+1}=\theta_t-\eta g_t}
$$

Where:

- $\theta_t$: Current parameters.
- $\eta$: Learning rate.
- $g_t=\nabla_\theta L(\theta_t)$: Current gradient.
- $\theta_{t+1}$: Updated parameters.

The learning rate controls the size of each update. A small learning rate may result in slow convergence, while an excessively large learning rate may cause oscillations or divergence.

**2. Momentum**

**Momentum** improves Gradient Descent by incorporating information from previous gradients rather than relying exclusively on the current gradient.

It maintains an exponential moving average of gradients:

$$
\boxed{m_t=\beta m_{t-1}+(1-\beta)g_t}
$$

The parameters are then updated using:

$$
\boxed{\theta_{t+1}=\theta_t-\eta m_t}
$$

Where:

- $m_t$: Moving average of gradients.
- $\beta$: Momentum coefficient, commonly 0.9.
- $g_t$: Current gradient.
- $\eta$: Learning rate.

Momentum helps accelerate optimization when gradients consistently point in the same direction and reduces oscillations when gradients alternate directions.

**Numerical Example — Momentum**

Suppose:

$$
\theta_0=2,\quad \eta=0.1,\quad \beta=0.9,\quad m_0=0
$$

**First iteration:** With gradient $g_1=4$:

$$
m_1=0.9(0)+0.1(4)=0.4
$$

$$
\theta_1=2-0.1(0.4)=\boxed{1.96}
$$

**Second iteration:** With gradient $g_2=3$:

$$
m_2=0.9(0.4)+0.1(3)=0.66
$$

$$
\theta_2=1.96-0.1(0.66)=\boxed{1.894}
$$

The second update incorporates information from both the previous and current gradients.

**3. RMSprop**

**RMSprop (Root Mean Square Propagation)** adapts the learning rate of each parameter based on the magnitude of its recent gradients.

It maintains an exponential moving average of squared gradients:

$$
\boxed{v_t=\beta v_{t-1}+(1-\beta)g_t^2}
$$

The parameter update is:

$$
\boxed{\theta_{t+1}=\theta_t-\frac{\eta}{\sqrt{v_t}+\epsilon}g_t}
$$

Where:

- $v_t$: Moving average of squared gradients.
- $\beta$: Decay coefficient, commonly 0.9.
- $\epsilon$: Small constant for numerical stability.

RMSprop reduces the effective learning rate for parameters with consistently large gradients and increases it relative to parameters with smaller gradients.

**Numerical Example — RMSprop**

Suppose:

$$
\theta_0=2,\quad \eta=0.1,\quad \beta=0.9,\quad v_0=0
$$

Assume $\epsilon$ is sufficiently small to neglect in the calculations.

**First iteration:** With gradient $g_1=4$:

$$
v_1=0.9(0)+0.1(4^2)=1.6
$$

$$
\theta_1=2-\frac{0.1(4)}{\sqrt{1.6}}
$$

$$
\boxed{\theta_1\approx1.6838}
$$

**Second iteration:** With gradient $g_2=3$:

$$
v_2=0.9(1.6)+0.1(3^2)=2.34
$$

$$
\theta_2=1.6838-\frac{0.1(3)}{\sqrt{2.34}}
$$

$$
\boxed{\theta_2\approx1.4877}
$$

Unlike Momentum, RMSprop uses squared gradients to adjust the magnitude of parameter updates.

**4. Adam**

**Adam (Adaptive Moment Estimation)** combines the main ideas of Momentum and RMSprop by maintaining two moving averages.

**First moment — Momentum-like behavior:**

$$
\boxed{m_t=\beta_1m_{t-1}+(1-\beta_1)g_t}
$$

**Second moment — RMSprop-like behavior:**

$$
\boxed{v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2}
$$

Where:

- $m_t$: Moving average of gradients.
- $v_t$: Moving average of squared gradients.
- $\beta_1$: First-moment decay coefficient, commonly 0.9.
- $\beta_2$: Second-moment decay coefficient, commonly 0.999.

Because these moving averages are initialized at zero, Adam applies bias correction:

$$
\hat m_t=\frac{m_t}{1-\beta_1^t},
\qquad
\hat v_t=\frac{v_t}{1-\beta_2^t}
$$

The final update rule is:

$$
\boxed{
\theta_{t+1}
=
\theta_t-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}
}
$$

Adam combines the directional information from Momentum with the adaptive update sizes of RMSprop.

**Numerical Example — Adam**

Suppose:

$$
\theta_0=2,\quad \eta=0.1,\quad \beta_1=0.9,\quad \beta_2=0.999
$$

With $m_0=v_0=0$ and negligible $\epsilon$.

**First iteration:** With gradient $g_1=4$:

Calculate the first and second moments:

$$
m_1=0.9(0)+0.1(4)=0.4
$$

$$
v_1=0.999(0)+0.001(4^2)=0.016
$$

Apply bias correction:

$$
\hat m_1=\frac{0.4}{1-0.9}=4
$$

$$
\hat v_1=\frac{0.016}{1-0.999}=16
$$

Update the parameter:

$$
\theta_1=2-0.1\frac{4}{\sqrt{16}}
$$

$$
\boxed{\theta_1=1.9}
$$

**Second iteration:** With gradient $g_2=3$:

$$
m_2=0.9(0.4)+0.1(3)=0.66
$$

$$
v_2=0.999(0.016)+0.001(3^2)=0.024984
$$

Apply bias correction:

$$
\hat m_2=\frac{0.66}{1-0.9^2}\approx3.4737
$$

$$
\hat v_2=\frac{0.024984}{1-0.999^2}\approx12.4982
$$

Update:

$$
\theta_2=1.9-0.1\frac{3.4737}{\sqrt{12.4982}}
$$

$$
\boxed{\theta_2\approx1.8017}
$$

Adam incorporates the history of gradients through the first moment and adjusts parameter updates using the second moment.

**5. Comparison of Optimization Algorithms**

| Characteristic | Gradient Descent | Momentum | RMSprop | Adam |
|---|---|---|---|---|
| **Uses current gradient** | Yes | Yes | Yes | Yes |
| **Uses previous gradients** | No | Yes | Through squared gradients | Yes |
| **Adaptive learning rate** | No | No | Yes | Yes |
| **Reduces oscillations** | Limited | Yes | Often | Often |
| **Main advantage** | Simple implementation | Accelerates convergence | Adapts parameter updates | Combines momentum and adaptive scaling |
| **Main limitation** | Can converge slowly | Requires tuning momentum and learning rate | Sensitive to hyperparameters | Higher computational and memory requirements |

**Numerical Comparison**

Using the same initial parameter $\theta_0=2$, learning rate $\eta=0.1$, and illustrative gradients $g_1=4$ and $g_2=3$:

| Optimizer | Initial parameter | Iteration 1 | Iteration 2 |
|---|---:|---:|---:|
| Momentum | 2.0000 | 1.9600 | 1.8940 |
| RMSprop | 2.0000 | 1.6838 | 1.4877 |
| Adam | 2.0000 | 1.9000 | 1.8017 |

These results demonstrate how the algorithms update parameters differently. Larger updates do not necessarily imply faster convergence or better optimization.

**6. Epochs, Batches, and Iterations**

Optimization is performed repeatedly during neural network training.

- **Epoch:** One complete pass through the training dataset.
- **Batch:** A subset of training observations processed together.
- **Iteration:** One parameter update using a batch.

For example, with 1,000 training observations and a batch size of 100:

$$
\text{Iterations per epoch}=\frac{1000}{100}=10
$$

Training for 50 epochs produces:

$$
\boxed{10\times50=500\text{ parameter updates}}
$$

**In summary:** Gradient Descent updates parameters using the current gradient, Momentum incorporates previous gradients, RMSprop adapts learning rates using squared gradients, and Adam combines momentum-based and adaptive optimization techniques.


![Backpropagation Pipeline](../results/07_autoencoder_vae/figures/optimization.png)

## **6. Autoencoder**

### **6.1 Notation**

An **Autoencoder (AE)** is an unsupervised representation-learning neural network designed to learn a compressed representation of input data and reconstruct the original input from that representation.

Unlike traditional supervised learning, an Autoencoder does not require class labels. Instead, it uses the input itself as the target.

Its objective is to learn a function:

$$
\boxed{\hat{x}=f_\theta(x)}
$$

Where:

- $x$: Original input.
- $\hat{x}$: Reconstructed input.
- $f_\theta$: Autoencoder parameterized by weights and biases.

The goal is to produce a reconstruction $\hat{x}$ that is as similar as possible to $x$.

---

**1. Autoencoder Architecture**

An Autoencoder consists of three main components:

**1.1 Encoder**

The **Encoder** transforms the original input into a lower-dimensional representation.

$$
\boxed{z=f_\phi(x)}
$$

Where:

- $x\in\mathbb{R}^{p}$: Original input containing $p$ features.
- $z\in\mathbb{R}^{k}$: Latent representation.
- $f_\phi$: Encoder function.
- $k$: Latent dimensionality, often $k<p$.

For example, if our dataset contains 10 numerical features, the Encoder could transform them into 3 latent features:

$$
\mathbb{R}^{10}\longrightarrow\mathbb{R}^{3}
$$

Unlike PCA, which learns linear components, an Autoencoder with nonlinear activation functions can learn nonlinear representations.

**1.2 Latent Space (Bottleneck)**

The **latent space** is the intermediate representation learned by the Encoder.

It contains information that the Decoder uses to reconstruct the input.

The bottleneck encourages the network to preserve important information rather than simply copying every input feature directly.

For example:

$$
x=[x_1,x_2,\ldots,x_{10}]
$$

The Encoder might produce:

$$
z=[0.72,-1.15,0.38]
$$

These three values represent learned combinations of the original features. They do not necessarily correspond to individual original variables.

Importantly, a small latent space does not guarantee meaningful representations by itself. The architecture, training objective, and regularization also influence what the network learns.

**1.3 Decoder**

The **Decoder** attempts to reconstruct the original input from the latent representation.

$$
\boxed{\hat{x}=g_\psi(z)}
$$

Where:

- $z$: Latent representation.
- $g_\psi$: Decoder function.
- $\hat{x}$: Reconstructed input.

Continuing the example:

$$
\mathbb{R}^{3}\longrightarrow\mathbb{R}^{10}
$$

The Decoder receives only the three latent features and attempts to recover all ten original features.

---

**2. How Does an Autoencoder Learn?**

An Autoencoder learns by minimizing the difference between the original input and its reconstruction.

For numerical features, a common reconstruction loss is **Mean Squared Error (MSE)**:

$$
\boxed{
L_{\text{AE}}=
\frac{1}{N}\sum_{i=1}^{N}
\frac{1}{p}\sum_{j=1}^{p}
(x_{ij}-\hat{x}_{ij})^2
}
$$

Where:

- $N$: Number of observations.
- $p$: Number of input features.
- $x_{ij}$: Original value of feature $j$ for observation $i$.
- $\hat{x}_{ij}$: Reconstructed value.

**Numerical Example**

Suppose an observation contains three features:

$$
x=[2,4,6]
$$

The Autoencoder produces:

$$
\hat{x}=[2.2,3.8,5.5]
$$

The reconstruction error is:

$$
L=\frac{(2-2.2)^2+(4-3.8)^2+(6-5.5)^2}{3}
$$

$$
L=\frac{0.04+0.04+0.25}{3}
$$

$$
\boxed{L=0.11}
$$

During training, backpropagation calculates gradients of this loss, and an optimizer such as Adam updates the weights to reduce reconstruction errors.

---

**3. Training Pipeline**

An Autoencoder is trained through the following steps:

1. **Input:** Provide a batch of observations $x$.
2. **Encoding:** Transform $x$ into a latent representation $z$.
3. **Decoding:** Reconstruct the input to obtain $\hat{x}$.
4. **Loss:** Calculate the reconstruction error $L(x,\hat{x})$.
5. **Backpropagation:** Compute the gradients of the loss.
6. **Optimization:** Update the weights and biases.

These steps are repeated across batches and epochs until training finishes.

---

**4. Autoencoders for Anomaly Detection**

One application of Autoencoders is **anomaly detection**.

The underlying idea is that an Autoencoder trained predominantly on normal observations may learn to reconstruct normal patterns more accurately than unusual ones.

For each observation, we can calculate an anomaly score:

$$
\boxed{
s(x_i)=\frac{1}{p}\sum_{j=1}^{p}(x_{ij}-\hat{x}_{ij})^2
}
$$

A larger reconstruction error may indicate an anomalous observation.

For example:

| Observation | Reconstruction MSE | Interpretation |
|---|---:|---|
| A | 0.012 | Low reconstruction error |
| B | 0.045 | Moderate reconstruction error |
| C | 1.240 | High reconstruction error |

If we establish an anomaly threshold using the 95th percentile of training reconstruction errors:

$$
\tau=Q_{0.95}(s_{\text{train}})
$$

Then:

$$
\boxed{
\operatorname{Anomaly}(x)=
\begin{cases}
1 & \text{if }s(x)>\tau\\
0 & \text{otherwise}
\end{cases}
}
$$

However, **high reconstruction error does not always mean an observation is anomalous**, and some anomalies may be reconstructed well. The effectiveness depends on the training data and learned representation.

---

**5. Autoencoder vs. PCA**

| Characteristic | PCA | Autoencoder |
|---|---|---|
| Learning approach | Linear dimensionality reduction | Neural representation learning |
| Nonlinear relationships | Not captured by standard PCA | Can capture them |
| Latent representation | Principal components | Learned latent features |
| Reconstruction | Linear projection | Learned decoder |
| Optimization | Eigenvalue decomposition / SVD | Usually gradient-based training |
| Complexity | Generally lower | Generally higher |

A linear Autoencoder with appropriate constraints and squared reconstruction loss can learn the same principal subspace as PCA, although its latent coordinates need not match PCA's components exactly.

**In summary:** An Autoencoder learns to compress input data into a latent representation and reconstruct it. By minimizing reconstruction error, it can learn useful representations for dimensionality reduction, feature extraction, and anomaly detection.


### **6.2 Implementation**

## **7. Variational Autoencoder (VAE)**

### **7.1 Notation**


**5. What is a Variational Autoencoder (VAE)?**

A **Variational Autoencoder (VAE)** is a probabilistic neural network that learns a latent representation of input data and reconstructs the original input from that representation.

Like a traditional Autoencoder (AE), a VAE consists of an **Encoder, a latent space, and a Decoder**.

However, the main difference is that **a VAE learns probability distributions in the latent space instead of directly learning a single latent vector for each observation**.

This allows a VAE to generate new observations by sampling from its learned latent distribution.

---

**1. VAE Architecture**

A VAE contains three main components.

**1.1 Encoder**

The Encoder transforms an input observation into the parameters of a probability distribution.

Instead of producing a single latent vector $z$, it produces:

- $\mu$: Mean vector.
- $\sigma^2$: Variance vector.

These parameters define a Gaussian distribution:

$$
\boxed{q_\phi(z\mid x)=\mathcal{N}(\mu_\phi(x),\operatorname{diag}(\sigma_\phi^2(x)))}
$$

Where:

- $x$: Original observation.
- $z$: Latent representation.
- $q_\phi(z\mid x)$: Approximate posterior distribution learned by the Encoder.
- $\phi$: Encoder parameters.

For example, if the input has 10 features and the latent space has 3 dimensions:

$$
\mathbb{R}^{10}\longrightarrow(\mu,\sigma^2)\in\mathbb{R}^{3}\times\mathbb{R}^{3}
$$

The Encoder produces two vectors, each containing three values.

**1.2 Latent Space and Sampling**

Unlike a traditional Autoencoder, a VAE does not use a fixed latent representation for each observation.

Instead, it samples a latent vector from the distribution estimated by the Encoder.

For example, suppose:

$$
\mu=[0.5,-0.2]
$$

$$
\sigma=[0.1,0.3]
$$

The latent vector is sampled from:

$$
z\sim\mathcal{N}(\mu,\operatorname{diag}(\sigma^2))
$$

However, directly sampling $z$ would make ordinary backpropagation through the sampling operation difficult.

To solve this, VAEs use the **Reparameterization Trick**.

$$
\boxed{z=\mu+\sigma\odot\epsilon}
$$

Where:

- $\mu$: Mean vector.
- $\sigma$: Standard deviation vector.
- $\epsilon\sim\mathcal{N}(0,I)$: Random noise.
- $\odot$: Element-wise multiplication.

**Numerical Example — Reparameterization**

Suppose the sampled noise is:

$$
\epsilon=[1,-0.5]
$$

Then:

$$
z=[0.5,-0.2]+[0.1,0.3]\odot[1,-0.5]
$$

$$
z=[0.5+0.1,-0.2-0.15]
$$

$$
\boxed{z=[0.6,-0.35]}
$$

The important idea is that the randomness is isolated in $\epsilon$, allowing gradients to propagate through $\mu$ and $\sigma$.

**1.3 Decoder**

The Decoder receives the sampled latent vector and reconstructs the original observation.

$$
\boxed{p_\theta(x\mid z)}
$$

This expression represents the probability distribution of the input conditioned on the latent variable.

For example, the Decoder might model a Gaussian distribution whose mean is the reconstruction:

$$
p_\theta(x\mid z)=\mathcal{N}(\mu_\theta(z),\sigma_x^2I)
$$

The reconstructed input can then be taken as:

$$
\hat{x}=\mu_\theta(z)
$$

Thus, the Decoder transforms the sampled latent representation into a reconstruction of the original features.

---

**2. How Does a VAE Learn?**

A VAE learns by optimizing two objectives simultaneously:

1. **Reconstruction:** Produce reconstructions similar to the original observations.
2. **Regularization:** Encourage the learned latent distributions to remain close to a predefined prior distribution.

The standard VAE loss is the negative Evidence Lower Bound (negative ELBO):

$$
\boxed{
L_{\text{VAE}}=
L_{\text{reconstruction}}+
D_{\mathrm{KL}}(q_\phi(z\mid x)\|p(z))
}
$$

Where:

- $L_{\text{reconstruction}}$: Negative expected log-likelihood of the original input under the Decoder.
- $D_{\mathrm{KL}}$: Kullback–Leibler divergence.
- $q_\phi(z\mid x)$: Distribution estimated by the Encoder.
- $p(z)$: Prior distribution, commonly $\mathcal{N}(0,I)$.

**2.1 Reconstruction Loss**

The reconstruction term measures how well the Decoder reconstructs the input.

For continuous numerical features, a common choice is MSE:

$$
\boxed{
L_{\text{reconstruction}}=
\frac{1}{p}\sum_{j=1}^{p}(x_j-\hat{x}_j)^2
}
$$

For example:

$$
x=[2,4,6],\qquad\hat{x}=[2.2,3.8,5.5]
$$

$$
L_{\text{reconstruction}}=
\frac{0.04+0.04+0.25}{3}
=\boxed{0.11}
$$

MSE is proportional to the negative log-likelihood under a Gaussian observation model with fixed variance. The exact scaling matters when combining it with the KL term.

**2.2 KL Divergence**

The **Kullback–Leibler divergence** measures how different the Encoder's latent distribution is from the prior distribution.

The standard prior is:

$$
\boxed{p(z)=\mathcal{N}(0,I)}
$$

For a diagonal Gaussian approximate posterior, KL divergence has a closed-form expression:

$$
\boxed{
D_{\mathrm{KL}}=
\frac{1}{2}\sum_{j=1}^{k}
\left(
\mu_j^2+\sigma_j^2-\log(\sigma_j^2)-1
\right)
}
$$

Where $k$ is the number of latent dimensions.

**Numerical Example — KL Divergence**

Suppose the Encoder produces a one-dimensional latent distribution with:

$$
\mu=0.5,\qquad\sigma=1
$$

Then:

$$
D_{\mathrm{KL}}=
\frac12(0.5^2+1^2-\log(1)-1)
$$

$$
=\frac12(0.25)=\boxed{0.125}
$$

A KL divergence of zero means the approximate posterior matches the prior.

The KL term encourages the latent distributions to remain organized around the prior, making sampling and generation possible.

**2.3 Combining Both Losses**

A commonly used variation is the **β-VAE**, which introduces a coefficient controlling the strength of regularization:

$$
\boxed{
L_{\beta\text{-VAE}}=
L_{\text{reconstruction}}+
\beta D_{\mathrm{KL}}
}
$$

- Smaller $\beta$: Places relatively more emphasis on reconstruction.
- Larger $\beta$: Places relatively more emphasis on matching the latent prior.

For example, suppose:

$$
L_{\text{reconstruction}}=0.11,\qquad D_{\mathrm{KL}}=0.125
$$

With $\beta=1$:

$$
L_{\text{VAE}}=0.11+0.125=\boxed{0.235}
$$

This is an illustrative combination of loss terms; in an actual implementation, their normalization and probabilistic interpretation must be consistent.

---

**3. VAE Training Pipeline**

A VAE is trained through the following steps:

1. **Input:** Provide a batch of observations $x$.
2. **Encoding:** Calculate the latent mean $\mu$ and variance $\sigma^2$.
3. **Sampling:** Generate $z=\mu+\sigma\odot\epsilon$.
4. **Decoding:** Reconstruct the input using the sampled latent vector.
5. **Loss:** Calculate reconstruction loss and KL divergence.
6. **Backpropagation:** Compute gradients through the reparameterized network.
7. **Optimization:** Update the parameters using an optimizer such as Adam.

These steps are repeated across batches and epochs.

---

**4. Variational Autoencoders for Anomaly Detection**

VAEs can also be used for **anomaly detection**.

A VAE trained predominantly on normal observations may learn a probability model that represents normal patterns better than unusual ones.

Several anomaly scores are possible.

**4.1 Reconstruction Error**

Similar to an Autoencoder:

$$
\boxed{
s_{\text{reconstruction}}(x_i)=
\frac1p\sum_{j=1}^{p}(x_{ij}-\hat{x}_{ij})^2
}
$$

A high reconstruction error may indicate an anomaly.

Because the VAE samples latent vectors, the reconstruction error can vary between samples. We can instead use the mean reconstruction or average the error over multiple latent samples.

**4.2 KL Divergence Score**

Another possibility is to measure how different an observation's approximate posterior is from the prior:

$$
\boxed{
s_{\text{KL}}(x_i)=
D_{\mathrm{KL}}(q_\phi(z\mid x_i)\|p(z))
}
$$

An unusually large KL divergence can indicate that an observation requires a latent distribution far from the prior.

However, KL divergence alone is not a universally reliable anomaly score.

**4.3 Combined Score**

We can also combine reconstruction error and KL divergence:

$$
\boxed{
s(x_i)=s_{\text{reconstruction}}(x_i)+
\lambda s_{\text{KL}}(x_i)
}
$$

Where $\lambda$ controls the relative contribution of the two components.

As with an Autoencoder, an empirical threshold can be established:

$$
\boxed{\tau=Q_{0.95}(s_{\text{train}})}
$$

An observation is flagged when:

$$
s(x_i)>\tau
$$

The score and threshold should be evaluated carefully because some anomalous observations may still receive low reconstruction errors or plausible latent distributions.

---

**5. Autoencoder vs. Variational Autoencoder**

| Characteristic | Autoencoder (AE) | Variational Autoencoder (VAE) |
|---|---|---|
| Learning approach | Deterministic representation learning | Probabilistic representation learning |
| Encoder output | Latent vector $z$ | Distribution parameters $\mu,\sigma^2$ |
| Latent representation | Deterministic | Sampled from a learned distribution |
| Decoder | Reconstructs the input | Models a conditional distribution of the input |
| Training loss | Reconstruction error | Reconstruction likelihood + KL regularization |
| Sampling | No explicit prior for generation | Supports sampling from a latent prior |
| Training complexity | Generally lower | Generally higher |
| Anomaly detection | Reconstruction error | Reconstruction, KL, or probabilistic scores |

**In summary:** A Variational Autoencoder extends the traditional Autoencoder by learning probability distributions in the latent space. Its Encoder estimates the mean and variance of a latent distribution, the reparameterization trick enables differentiable sampling, and the Decoder reconstructs the input. By combining reconstruction quality with KL regularization, a VAE learns a probabilistically structured latent space useful for representation learning, data generation, and anomaly detection.


![Backpropagation Pipeline](../results/07_autoencoder_vae/figures/vae.png)